# Training the YOLOv8 defect detector

Renders the full `.xyz` dataset, splits the Label Studio-annotated
`data/yolo_labeled/` set into train/test, and fine-tunes a YOLOv8 model on it.

Requires `ultralytics` (and a GPU for reasonable training time — the original
training was run on Google Colab's free tier).

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
logging.basicConfig(level=logging.INFO)

from graphene_defect_detector.config import PipelineConfig
from graphene_defect_detector import rendering, detection, splitting

config = PipelineConfig.from_yaml("../configs/pipeline.yaml")
config.ensure_output_dirs()

## 1. Render the structures used for inference/training imagery

In [ ]:
rendering.render_dataset(config.xyz_dir, config.renders_dir, config.render_resolution)

## 2. Split the labeled dataset into train/ and test/

`data/yolo_labeled/` holds Label Studio-exported YOLO-format labels + images in a flat
directory. This creates `train/` and `test/` subfolders inside it, cross-platform (no
shelling out to `mv`).

In [ ]:
train_dir, test_dir = splitting.split_yolo_dataset(config.yolo_labeled_dir, test_fraction=0.2, seed=0)
train_dir, test_dir

## 3. Generate a dataset yaml with an absolute `path:`

Ultralytics resolves a relative `path:` against its own global `datasets_dir`
setting rather than the yaml file's location, so we write out a resolved copy.

In [ ]:
dataset_yaml = detection.write_dataset_yaml(
    config.yolo_labeled_dir, "../configs/yolo_dataset.resolved.yaml",
)
dataset_yaml

## 4. Train

In [ ]:
model = detection.train(
    dataset_yaml=dataset_yaml,
    weights_path="yolov8n.pt",  # or config.weights_path to fine-tune further
    epochs=100,
)

In [ ]:
metrics = model.val()
metrics

In [ ]:
model.export(format="onnx")